## Basic RAG with Ollama + ChromaDB

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline
using local Ollama models and ChromaDB as the vector store.


In [1]:
import os
import sys
import subprocess
import time

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q
    !curl -fsSL https://ollama.com/install.sh | sh
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5)
    !ollama pull nomic-embed-text-v2-moe:latest
    !ollama pull llama3.1:8b-instruct-q4_K_M

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import os
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import PROMPT_V1, PROMPT_V2, build_rag_chain
from reliablerag.env import load_secrets
from reliablerag.experiment import evaluate_results, run_rag_experiment
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import get_hyde_retriever, get_hybrid_retriever, get_hybrid_reranked_retriever, get_or_build_vector_store, get_reranked_retriever, get_reranker, get_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [4]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
GENERATOR_MODEL    = os.environ["GENERATOR_MODEL"]
JUDGE_MODEL        = os.environ["JUDGE_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generator model : {GENERATOR_MODEL}")
print(f"Judge model     : {JUDGE_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

Provider        : ollama
Embedding model : nomic-embed-text-v2-moe:latest
Generator model : mistral-small3.2:24b
Judge model     : llama3.1:8b-instruct-q4_K_M
Chroma dir      : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [5]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)

# Generator: large model used to write the final answer.
llm = create_llm(PROVIDER, GENERATOR_MODEL)

# Judge: smaller model used by the TRACe eval. Deterministic decoding so
# repeat evals on the same inputs don't drift. Bumped to a smaller open-source
# model (Llama 3.1 8B) — gemma 12B as judge was making each eval call ~3-4min.
judge_llm = create_llm(PROVIDER, JUDGE_MODEL, temperature=0)

# Cross-encoder reranker. Built once outside the per-sample loop — the
reranker = get_reranker()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### 2. Load CUAD Samples from RAGBench

Each sample contains a legal contract (`documents`), a `question`, a reference `response` generated
by Claude 3 Haiku, and pre-computed **TRACe labels** annotated by GPT-4:
`adherence`, `context_relevance`, `utilization`, `completeness`.

In [6]:
N_SAMPLES = 20

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))


def fmt(v):
    return f"{v:.3f}" if v is not None else "N/A"


print(f"Loaded {len(samples)} CUAD samples")

s = samples[0]
print(f"\nQuestion         : {s['question']}")
print(f"Doc length       : {len(s['documents'][0])} chars")
print(f"Adherence score  : {s['adherence_score']}")
print(f"Relevance score  : {fmt(s['relevance_score'])}")
print(f"Utilization score: {fmt(s['utilization_score'])}")
print(f"Completeness     : {fmt(s['completeness_score'])}")
print(f"\nAlso available   : ragas_faithfulness={fmt(s['ragas_faithfulness'])}, "
      f"trulens_groundedness={fmt(s['trulens_groundedness'])}")

Loaded 20 CUAD samples

Question         : Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Doc length       : 122054 chars
Adherence score  : True
Relevance score  : 0.000
Utilization score: 0.000
Completeness     : 1.000

Also available   : ragas_faithfulness=N/A, trulens_groundedness=N/A


In [61]:
df_preview = dataset.to_pandas()
df_preview.head(10)
df_preview.get("question").get(1528)
# len(df_preview.get("question"))

'Does one party have the right to terminate or is consent or notice required of the counterparty if such party undergoes a change of control, such as a merger, stock sale, transfer of all or substantially all of its assets or business, or assignment by operation of law?'

### 3. Run RAG on Each Sample

CUAD contracts are up to 11k tokens each, so we chunk each document before indexing.
We build a fresh ephemeral vector store per sample (CUAD has 1 doc per question, so no cross-contamination).

In [8]:
# Exp E — Dense-only baseline (cosine, nomic, chunk_size=500, top_k=20, N=20).
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_e = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_retriever(vs, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
)


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.903s  (291 chunks embedded)
[timing] retrieve : 0.171s
[timing] retriever: 0.119s
[timing] llm      : 21.535s
  our: I do not know. The provided text does not contain any information regarding source code deposits, escrow agreements, or terms for their release due to bankruptcy or insolvency.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.626s  (47 chunks embedded)
[timing] retrieve : 0.142s
[timing] retriever: 0.110s
[timing] llm      : 27.013s
  our: The provided text does not mention a license granted by one party to its counterparty. It contains only language regarding the execu

In [9]:
# Exp E — TRACe evaluation (dense-only baseline)
JUDGE_N_RUNS = 3
agg_e = evaluate_results(results_e, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Exp E (dense baseline) — Rel {agg_e['avg_relevance']:.3f} / Util {agg_e['avg_utilization']:.3f} / Comp {agg_e['avg_completeness']:.3f} / Adh {agg_e['adherence_rate']:.0%}")
print(f"Ref   (GPT-4 labels)   — Rel 0.069 / Util 0.042 / Comp 0.717 / Adh 90%")

[1/20] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The response claims that there 
  Relevance   : 0.072 — The provided documents appear to be a contract or agreement between two parties, outlining
  Utilization : 0.072
  Completeness: 1.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.560 — The provided documents contain information about the execution of the contract in counterp
  Utilization : 0.056
  Completeness: 0.100

[3/20] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.027 — The provided documents contain useful information for answering the question 

In [10]:
# Experiment G — Hybrid retrieval (BM25 + dense, RRF fusion).
# Hypothesis: BM25 recovers exact-term clause misses that dense retrieval drops.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_g = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hybrid_retriever(vs, chunks, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
)


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.383s  (291 chunks embedded)
[timing] retrieve : 0.148s
[timing] retriever: 0.109s
[timing] retriever: 0.003s
[timing] llm      : 30.083s
  our: The provided context does not contain any information regarding source code or escrow agreements. Therefore, I do not know if one party is required to deposit its source code into escrow.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 1.849s  (47 chunks embedded)
[timing] retrieve : 0.151s
[timing] retriever: 0.115s
[timing] retriever: 0.000s
[timing] llm      : 24.323s
  our: Based on the provided excerpts, there is no mention of a license be

In [11]:
# Experiment G — TRACe evaluation
JUDGE_N_RUNS = 3
agg_g = evaluate_results(results_g, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment G (hybrid, bm25=0.5)   — Rel {agg_g['avg_relevance']:.3f} / Util {agg_g['avg_utilization']:.3f} / Comp {agg_g['avg_completeness']:.3f} / Adh {agg_g['adherence_rate']:.0%}")

[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence in the respo
  Relevance   : 0.322 — The relevant information for answering this question can be found in Section 3 of the Moel
  Utilization : 0.093
  Completeness: 0.289

[2/20] [FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The claim that there is no ment
  Relevance   : 0.654 — The relevant information for answering this question can be found in document 1, which con
  Utilization : 0.153
  Completeness: 0.234

[3/20] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.063 — The relevant information for answering this question can be found in Section 

In [12]:
# Experiment H — Tune RRF weights (bm25_weight=0.3, dense_weight=0.7).
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
BM25_WEIGHT    = 0.3
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

results_h = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hybrid_retriever(vs, chunks, top_k=TOP_K, bm25_weight=BM25_WEIGHT),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
)


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.684s  (291 chunks embedded)
[timing] retrieve : 0.147s
[timing] retriever: 0.119s
[timing] retriever: 0.003s
[timing] llm      : 28.675s
  our: I do not know the answer to this question because the provided text does not contain any information regarding source code or escrow requirements.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.813s  (47 chunks embedded)
[timing] retrieve : 0.163s
[timing] retriever: 0.122s
[timing] retriever: 0.000s
[timing] llm      : 25.117s
  our: Based on the provided text, there is no mention of a license granted by one party to its counterparty. The c

In [13]:
# Experiment H — TRACe evaluation (bm25_weight=0.3)
JUDGE_N_RUNS = 3
agg_h = evaluate_results(results_h, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment H  (bm25={BM25_WEIGHT})          — Rel {agg_h['avg_relevance']:.3f} / Util {agg_h['avg_utilization']:.3f} / Comp {agg_h['avg_completeness']:.3f} / Adh {agg_h['adherence_rate']:.0%}")

[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The relevant information for
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported because it correctly states that there is no mention 
  Relevance   : 0.072 — The contract primarily covers terms regarding property disposition, notice requirements, a
  Utilization : 0.072
  Completeness: 1.000

[3/20] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.035 — The relevant information for answering this question can be found in Section 4a of documen
  Utilization : 0.035
  Completeness: 1.000

[4/20] [FAIL] Does the contract

In [14]:
# Experiment I — Reranker on top of equal-weight hybrid.
CHUNK_SIZE, CHUNK_OVERLAP, FETCH_K, TOP_N = 500, 50, 40, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

reranker = get_reranker()

results_i = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hybrid_reranked_retriever(vs, chunks, reranker, fetch_k=FETCH_K, top_n=TOP_N),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"retrieve+rerank (fetch={FETCH_K}→top-{TOP_N})",
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.613s  (291 chunks embedded)
[timing] retrieve+rerank (fetch=40→top-20) : 0.706s
[timing] retriever: 0.142s
[timing] retriever: 0.003s
[timing] llm      : 37.402s
  our: I do not have information regarding whether a party is required to deposit its source code into escrow, as the provided context does not mention "source code" or "escrow."
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.803s  (47 chunks embedded)
[timing] retrieve+rerank (fetch=40→top-20) : 0.723s
[timing] retriever: 0.149s
[timing] retriever: 0.000s
[timing] llm      : 21.645s
  our: The provided contract excerpts do

In [15]:
# Experiment I — TRACe evaluation (hybrid + cross-encoder reranker)
JUDGE_N_RUNS = 3
agg_i = evaluate_results(results_i, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment I (hybrid+reranker)    — Rel {agg_i['avg_relevance']:.3f} / Util {agg_i['avg_utilization']:.3f} / Comp {agg_i['avg_completeness']:.3f} / Adh {agg_i['adherence_rate']:.0%}")

[1/20] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is not supported by the documents. The response claims that there 
  Relevance   : 0.102 — The relevant information for answering this question can be found in Section 1 of the Agre
  Utilization : 0.000
  Completeness: 0.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.163 — The contract excerpts provided contain information about standard legal provisions, notice
  Utilization : 0.163
  Completeness: 1.000

[3/20] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.040 — The relevant document for answering this question is Section 2a, which discus

In [16]:
# Experiment J — HyDE (Hypothetical Document Embeddings).
# Vocabulary mismatch root cause: embed a hypothetical clause instead of the raw query.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_j = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.851s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 1.862s
[timing] llm      : 1.673s
[timing] llm      : 23.464s
  our: I do not know the answer to this question because the provided text does not mention source code, escrow arrangements, or terms related to bankruptcy or insolvency.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.874s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 1.486s
[timing] llm      : 1.025s
[timing] llm      : 19.638s
  our: No, the provided context does not contain any information regarding a license granted by one party to another. The t

In [17]:
# Experiment J — TRACe evaluation (HyDE)
JUDGE_N_RUNS = 3
agg_j = evaluate_results(results_j, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment J  (HyDE)              — Rel {agg_j['avg_relevance']:.3f} / Util {agg_j['avg_utilization']:.3f} / Comp {agg_j['avg_completeness']:.3f} / Adh {agg_j['adherence_rate']:.0%}")

[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The provided documents appea
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response,
  Relevance   : 1.000 — The provided context consists of multiple instances of the same clause stating that headin
  Utilization : 0.100
  Completeness: 0.100

[3/20] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.557 — The provided documents contain information about the execution and effectiveness of the ag
  Utilization : 0.084
  Completeness: 0.150

[4/20] [FAIL] Does the contract

In [18]:
# Experiment K — Purpose-built sentence encoder (BAAI/bge-large-en-v1.5) + HyDE.
# Hypothesis: bge-large is contrastive-trained for cosine retrieval — better geometry than nomic.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter          = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embeddings_bge    = create_embeddings("huggingface", "BAAI/bge-large-en-v1.5")
COLLECTION_TAG_M1 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_bge"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_k = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings_bge, hyde_llm, top_k=TOP_K),
    embeddings=embeddings_bge,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_M1,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]


[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.086s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 2.523s
[timing] llm      : 1.844s
[timing] llm      : 24.659s
  our: I do not have enough information in the provided context to determine if a party is required to deposit source code into escrow. The text mentions bankruptcy and insolvency as conditions for termination, but it does not mention "source code" or "escrow."
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.924s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 1.142s
[timing] llm      : 2.305s
[timing] llm      : 20.829s
  our: Based on the provided text

In [19]:
# Experiment K — TRACe evaluation (bge-large-en-v1.5 + HyDE)
JUDGE_N_RUNS = 3
agg_k = evaluate_results(results_k, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment K (bge-large + HyDE)  — Rel {agg_k['avg_relevance']:.3f} / Util {agg_k['avg_utilization']:.3f} / Comp {agg_k['avg_completeness']:.3f} / Adh {agg_k['adherence_rate']:.0%}")

[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The relevant information for
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — Claim a: The response states that there is no mention of a license granted by one party to
  Relevance   : 0.106 — The relevant document is 1a, which contains boilerplate provisions such as amendments and 
  Utilization : 0.106
  Completeness: 1.000

[3/20] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — The response as a whole is supported by the documents. The first sentence in the response 
  Relevance   : 0.082 — The relevant information for answering this question can be found in sections 0a, 1a, and 
  Utilization : 0.082
  Completeness: 1.000

[4/20] [FAIL] Does the contract

In [20]:
# Experiment L — Legal-domain BERT (nlpaueb/legal-bert-base-uncased) + HyDE.
# Hypothesis: legal-bert vocabulary matches CUAD contracts better despite CLS-pooling.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter          = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
embeddings_legal  = create_embeddings("huggingface", "nlpaueb/legal-bert-base-uncased")
COLLECTION_TAG_M2 = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}_legalbert"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_l = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings_legal, hyde_llm, top_k=TOP_K),
    embeddings=embeddings_legal,
    generator_llm=create_llm(PROVIDER, "gemma4:12b-it-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG_M2,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 1.912s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 2.255s
[timing] llm      : 2.027s
[timing] llm      : 24.557s
  our: I do not have enough information to answer this question based on the provided context. The retrieved text does not mention source code, escrow agreements, or provisions regarding bankruptcy or insolvency.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 1.129s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 1.154s
[timing] llm      : 1.070s
[timing] llm      : 26.336s
  our: Based on the provided text, there is no mention of a license granted by one

In [21]:
# Experiment L — TRACe evaluation (legal-bert-base-uncased + HyDE)
JUDGE_N_RUNS = 3
agg_l = evaluate_results(results_l, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment L (legal-bert + HyDE) — Rel {agg_l['avg_relevance']:.3f} / Util {agg_l['avg_utilization']:.3f} / Comp {agg_l['avg_completeness']:.3f} / Adh {agg_l['adherence_rate']:.0%}")

[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is supported by the documents. The first sentence of the response 
  Relevance   : 0.049 — The relevant documents for answering this question are those that discuss confidentiality,
  Utilization : 0.049
  Completeness: 1.000

[2/20] [FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence in the respo
  Relevance   : 0.053 — The relevant information for answering this question can be found in document 1, which out
  Utilization : 0.053
  Completeness: 1.000

[3/20] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The document that contains u
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[4/20] [FAIL] Does the contract

In [22]:
# Experiment M — HyDE + nomic + PROMPT_V1 + mistral-small3.2:24b.
# Isolates model effect: same retrieval and prompt as Exp J, only generator changes.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_m = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "mistral-small3.2:24b"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)



[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 4.604s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 2.120s
[timing] llm      : 2.092s
[timing] llm      : 30.329s
  our: I don't know. The provided context does not mention anything about depositing source code into escrow or related events like bankruptcy or insolvency.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.774s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 4.240s
[timing] llm      : 2.096s
[timing] llm      : 14.108s
  our: I don't know. The provided context does not mention anything about a license.
  ref: No, the contract does not contain a license g

In [23]:
# Experiment M — TRACe evaluation (HyDE + PROMPT_V1 + mistral-small3.2:24b)
JUDGE_N_RUNS = 3
agg_m = evaluate_results(results_m, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment M  (HyDE + PROMPT_V1 + mistral-small3.2) — Rel {agg_m['avg_relevance']:.3f} / Util {agg_m['avg_utilization']:.3f} / Comp {agg_m['avg_completeness']:.3f} / Adh {agg_m['adherence_rate']:.0%}")


[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence 'I don't kno
  Relevance   : 1.000 — The provided context contains useful information for answering the question, specifically 
  Utilization : 1.000
  Completeness: 1.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The first sentence in the response,
  Relevance   : 1.000 — The provided context contains multiple instances of the same information, which is useful 
  Utilization : 0.000
  Completeness: 0.000

[3/20] [PASS] The date when the contract is effective ...
  Adherence   : PASS  — The response as a whole is supported by the documents because it accurately reflects that 
  Relevance   : 0.557 — The provided context contains multiple documents that discuss the execution a

In [39]:
# Experiment N — HyDE + nomic + PROMPT_V2 + mistral-small3.2:24b.
# Isolates prompt effect on top of Exp M: same model and retrieval, new prompt.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_n = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "mistral-small3.2:24b"),
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)



[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.459s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 7.437s
[timing] llm      : 2.299s
[timing] llm      : 32.417s
  our: NO. The contract does not contain any clause requiring a party to deposit its source code into escrow with a third party for release under specific events like bankruptcy or insolvency. The provided context only discusses breach of terms and conditions, written notice, and cure periods, as well as immediate termination upon bankruptcy or insolvency, but it does not mention anything about source code escrow.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 2.

In [40]:
# Experiment N — TRACe evaluation (HyDE + PROMPT_V2 + mistral-small3.2:24b)
JUDGE_N_RUNS = 3
agg_n = evaluate_results(results_n, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment N  (HyDE + PROMPT_V2 + mistral-small3.2) — Rel {agg_n['avg_relevance']:.3f} / Util {agg_n['avg_utilization']:.3f} / Comp {agg_n['avg_completeness']:.3f} / Adh {agg_n['adherence_rate']:.0%}")


[1/20] [PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response as a whole is supported by the documents because it accurately reflects the c
  Relevance   : 1.000 — The provided context only discusses breach of terms and conditions, written notice, and cu
  Utilization : 1.000
  Completeness: 1.000

[2/20] [PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — The response as a whole is supported by the documents. The claim that 'NO' is made in sent
  Relevance   : 1.000 — The provided context contains repetitive statements about headings in the agreement, which
  Utilization : 1.000
  Completeness: 1.000

[3/20] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The relevant document is 0a,
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[4/20] [FAIL] Does the contract

In [26]:
# Experiment O — HyDE + nomic + PROMPT_V1 + llama3.1:8b-instruct-q4_K_M.
# Isolates model effect vs Exp J (gemma4→llama3.1), same retrieval and prompt.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_o = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M"),
    prompt_template=PROMPT_V1,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)



[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.013s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 2.189s
[timing] llm      : 2.065s
[timing] llm      : 5.917s
  our: I don't know. The provided context does not mention anything about escrowing source code or specific release conditions such as bankruptcy or insolvency.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 0.629s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 1.937s
[timing] llm      : 1.178s
[timing] llm      : 3.874s
  our: No, the retrieved context does not mention a license being granted. It discusses how parties can change their address for service

In [27]:
# Experiment O — TRACe evaluation (HyDE + PROMPT_V1 + llama3.1:8b)
JUDGE_N_RUNS = 3
agg_o = evaluate_results(results_o, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment O  (HyDE + PROMPT_V1 + llama3.1:8b)      — Rel {agg_o['avg_relevance']:.3f} / Util {agg_o['avg_utilization']:.3f} / Comp {agg_o['avg_completeness']:.3f} / Adh {agg_o['adherence_rate']:.0%}")


[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence of the respo
  Relevance   : 0.709 — The provided context is relevant to answering the question as it pertains to engagement le
  Utilization : 0.071
  Completeness: 0.100

[2/20] [FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response as a whole is not supported by the documents. The first sentence in the respo
  Relevance   : 0.560 — The relevant information for answering this question is contained within the 'Counterparts
  Utilization : 0.056
  Completeness: 0.100

[3/20] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — parse error: Invalid json output: {
"relevance_explanation": "The provided documents appea
  Relevance   : 0.000 — 
  Utilization : 0.000
  Completeness: 0.000

[4/20] [FAIL] Does the contract

In [36]:
s# Experiment P — HyDE + nomic + PROMPT_V2 + llama3.1:8b-instruct-q4_K_M.
# Isolates prompt effect vs Exp O: same model and retrieval, new prompt.
CHUNK_SIZE, CHUNK_OVERLAP, TOP_K = 500, 50, 20
splitter       = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
COLLECTION_TAG = f"cs{CHUNK_SIZE}_co{CHUNK_OVERLAP}"

hyde_llm = create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M")  # HyDE doc generation; keep fixed across experiments for fair comparison
results_p = run_rag_experiment(
    samples,
    retriever_factory=lambda vs, chunks: get_hyde_retriever(vs, embeddings, hyde_llm, top_k=TOP_K),
    embeddings=embeddings,
    generator_llm=create_llm(PROVIDER, "llama3.1:8b-instruct-q4_K_M"),
    prompt_template=PROMPT_V2,
    splitter=splitter,
    persist_dir=CHROMA_PERSIST_DIR,
    collection_tag=COLLECTION_TAG,
    retrieve_label=f"hyde retrieve (top-{TOP_K})",
)



[1/20] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] vector store : 5.463s  (291 chunks embedded)
[timing] hyde retrieve (top-20) : 2.427s
[timing] llm      : 2.225s
[timing] llm      : 7.794s
  our: NO.

The contract text does not mention anything about depositing source code into escrow or releasing it under certain conditions. The repetitive text only mentions consequences for breaching parties when they become bankrupt or insolvent.
  ref: No, there is no requirement in the given agreement for one party to deposit its source code into escrow with a third party. The agreement does not mention anything about source code or an escrow arrangement.

[2/20] Does the contract contain a license granted by one party to its counterparty?...
[timing] vector store : 1.026s  (47 chunks embedded)
[timing] hyde retrieve (top-20) : 2.086s
[timing] llm      : 1.985s
[timing] llm      : 8.630s
  our: NO. The context provided appears to be rel

In [37]:
# Experiment P — TRACe evaluation (HyDE + PROMPT_V2 + llama3.1:8b)
JUDGE_N_RUNS = 3
agg_p = evaluate_results(results_p, judge_llm, n_runs=JUDGE_N_RUNS)
print(f"Experiment P  (HyDE + PROMPT_V2 + llama3.1:8b)      — Rel {agg_p['avg_relevance']:.3f} / Util {agg_p['avg_utilization']:.3f} / Comp {agg_p['avg_completeness']:.3f} / Adh {agg_p['adherence_rate']:.0%}")


[1/20] [FAIL] Is one party required to deposit its source code into escrow with a th...
  Adherence   : FAIL  — The response claims that the contract text does not mention anything about depositing sour
  Relevance   : 0.658 — The relevant information for answering this question can be found in all 19 documents, as 
  Utilization : 0.066
  Completeness: 0.100

[2/20] [FAIL] Does the contract contain a license granted by one party to its counte...
  Adherence   : FAIL  — The response claims that there is no specific text referring to a 'license' in the contrac
  Relevance   : 0.041 — The relevant document is 0 (or any other document with the same content, as they are ident
  Utilization : 0.041
  Completeness: 1.000

[3/20] [FAIL] The date when the contract is effective ...
  Adherence   : FAIL  — The response as a whole is partially supported by the documents. The claim that 'the contr
  Relevance   : 0.557 — The provided documents contain information about the execution and delivery o